# CNN–BiLSTM pause detection on Kaggle

This notebook trains a frame-level voice-activity/pause detector using a CNN frontend and bidirectional LSTM, replacing the original CNN–GRU sequence model. It is designed for a Kaggle GPU notebook.

## Expected raw-data layout

Kaggle extracts the uploaded dataset automatically. This notebook uses the extracted LibriParty directory directly:

```text
/kaggle/input/datasets/whitedevilanand/libriparty/LibriParty/dataset/
  train/session_0/session_0_mixture.wav
  train/session_0/session_0.json
  ...
```

The JSON parser below handles the LibriParty format, where utterance lists are stored under speaker IDs such as `"1455"`. If your annotation schema differs, edit only `read_speech_segments`.

The split is performed at session/group level, never at crop level, to prevent audio from one session appearing in both train and test.

In [ ]:
# Configuration: this is the extracted Kaggle dataset path.
from pathlib import Path
import json, random, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


REQUESTED_DATA_ROOT = Path('/kaggle/input/datasets/whitedevilanand/libriparty/LibriParty/dataset')
DATASET_CANDIDATES = [
    REQUESTED_DATA_ROOT,
    Path('/kaggle/input/libriparty/LibriParty/dataset'),
    Path('/kaggle/input/libriparty/dataset'),
]
DATASET_CANDIDATES += [p for p in Path('/kaggle/input').glob('**/dataset') if p not in DATASET_CANDIDATES]
DATA_ROOT = next((p for p in DATASET_CANDIDATES if (p / 'train').is_dir()), None)
WORK_ROOT = Path('/kaggle/working/cnn_lstm_vad')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SEED = 42
SAMPLE_RATE = 16_000
N_MELS = 40
N_FFT = 512
FRAME_LENGTH = int(SAMPLE_RATE * 0.025)
HOP_LENGTH = int(SAMPLE_RATE * 0.010)
CROP_FRAMES = 300       # about 3 seconds
EVAL_FRAMES = 600       # about 6 seconds; evaluation windows are non-random
BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7
NUM_WORKERS = 2
if DATA_ROOT is None: raise FileNotFoundError(
    'Could not find a Kaggle dataset directory containing train/. Checked: '
    + ', '.join(str(p) for p in DATASET_CANDIDATES)
)
print('Dataset root:', DATA_ROOT)
print('Output root:', WORK_ROOT)

In [ ]:
# Kaggle has already extracted the uploaded dataset; DATA_ROOT is configured above.
print('Using extracted dataset:', DATA_ROOT)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
    balanced_accuracy_score, confusion_matrix, classification_report)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('Device:', DEVICE, '| AMP:', USE_AMP)
if DEVICE.type == 'cuda': print(torch.cuda.get_device_name(0))

## Discover sessions and convert annotations to frame labels

In [ ]:
def read_speech_segments(json_path):
    """Return [(start_seconds, stop_seconds), ...] from a session JSON.

    Adapt this function if your uploaded annotation schema is not LibriParty-style.
    """
    with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
    raw = data.get('speech_segments', data.get('segments', data.get('speech')))
    if raw is None:
        # LibriParty JSON stores utterance lists under speaker IDs such as "1455".
        raw = [item for value in data.values() if isinstance(value, list) for item in value]
    segments = []
    for item in raw:
        if isinstance(item, dict):
            start = item.get('start', item.get('start_time'))
            stop = item.get('stop', item.get('end', item.get('end_time')))
        else:
            start, stop = item[0], item[1]
        if start is not None and stop is not None and float(stop) > float(start):
            segments.append((float(start), float(stop)))
    return segments


wav_files = sorted(DATA_ROOT.rglob('*.wav'))
records = []
for wav_path in wav_files:
    stem = wav_path.stem
    if stem.endswith('_mixture'): stem = stem[:-len('_mixture')]
    candidates = [
        wav_path.with_name(stem + '.json'),
        wav_path.with_name(stem + '.JSON'),
        wav_path.with_suffix('.json'),
        wav_path.with_suffix('.JSON'),
    ]
    json_path = next((p for p in candidates if p.exists()), None)
    if json_path is None: continue
    relative_parts = wav_path.relative_to(DATA_ROOT).parts
    group = '/'.join(relative_parts[-3:-1]) if len(relative_parts) >= 3 else wav_path.parent.name
    records.append({'wav': wav_path, 'json': json_path, 'group': group})
if not records:
    sample_wavs = [str(p) for p in wav_files[:5]]
    raise FileNotFoundError(
        f'No paired .wav/.json files found below {DATA_ROOT}. '
        f'Found {len(wav_files)} WAV files; examples: {sample_wavs}'
    )
manifest = pd.DataFrame(records)
print(f'Found {len(manifest)} annotated sessions.')
display(manifest.head())

In [ ]:
def extract_session(record):
    audio, sr = librosa.load(record['wav'], sr=SAMPLE_RATE, mono=True)
    peak = np.max(np.abs(audio))
    if peak > 1e-8: audio = audio / peak * 0.95
    mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=FRAME_LENGTH, n_mels=N_MELS,
        center=False, power=2.0)
    mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    n_frames = mel.shape[1]
    labels = np.zeros(n_frames, dtype=np.float32)
    frame_times = np.arange(n_frames) * HOP_LENGTH / SAMPLE_RATE
    for start, stop in read_speech_segments(record['json']):
        labels[(frame_times >= start) & (frame_times < stop)] = 1.0
    return {'mel': mel, 'labels': labels, 'group': record['group'], 'path': str(record['wav'])}


sessions = []
for i, row in manifest.iterrows():
    try: sessions.append(extract_session(row))
    except Exception as exc: warnings.warn(f'Skipping {row.wav}: {exc}')
if not sessions: raise RuntimeError('No sessions could be preprocessed.')
print('Frames:', sum(len(s['labels']) for s in sessions),
      '| speech fraction:', np.mean(np.concatenate([s['labels'] for s in sessions])))

In [ ]:
groups = np.array([s['group'] for s in sessions])
indices = np.arange(len(sessions))
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val, test_idx = next(gss.split(indices, groups=groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)
train_rel, val_rel = next(gss2.split(train_val, groups=groups[train_val]))
train_idx, val_idx = train_val[train_rel], train_val[val_rel]
train_sessions = [sessions[i] for i in train_idx]
val_sessions = [sessions[i] for i in val_idx]
test_sessions = [sessions[i] for i in test_idx]
print(f'Train/validation/test sessions: {len(train_sessions)}/{len(val_sessions)}/{len(test_sessions)}')

In [ ]:
class SessionDataset(Dataset):
    def __init__(self, sessions, crop_frames, train=False):
        self.sessions, self.crop_frames, self.train = sessions, crop_frames, train
        self.items = [(i, start) for i, s in enumerate(sessions)
                      for start in range(0, max(1, len(s['labels'])), crop_frames if train else crop_frames)]
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        i, start = self.items[index]; s = self.sessions[i]
        if self.train and len(s['labels']) > self.crop_frames:
            start = np.random.randint(0, len(s['labels']) - self.crop_frames + 1)
        x = s['mel'][:, start:start+self.crop_frames]
        y = s['labels'][start:start+self.crop_frames]
        valid = len(y)
        if x.shape[1] < self.crop_frames:
            pad = self.crop_frames - x.shape[1]
            x = np.pad(x, ((0, 0), (0, pad)), constant_values=0)
            y = np.pad(y, (0, pad), constant_values=-1)
        return torch.from_numpy(x[None]), torch.from_numpy(y), torch.tensor(valid)


train_ds = SessionDataset(train_sessions, CROP_FRAMES, train=True)
val_ds = SessionDataset(val_sessions, EVAL_FRAMES)
test_ds = SessionDataset(test_sessions, EVAL_FRAMES)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=USE_AMP)
train_loader = DataLoader(train_ds, shuffle=True, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)
print('Batches:', len(train_loader), len(val_loader), len(test_loader))

## CNN–BiLSTM model and training utilities

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(self, n_mels=N_MELS, channels=64, hidden=128, layers=2, dropout=0.25):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, channels, 3, padding=1), nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Dropout2d(dropout))
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(channels, hidden, num_layers=layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0, bidirectional=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden * 2), nn.Dropout(dropout), nn.Linear(hidden * 2, 1))
    def forward(self, x):
        h = self.freq_pool(self.frontend(x)).squeeze(2).transpose(1, 2)
        h, _ = self.rnn(h)
        return self.head(h).squeeze(-1)


def masked_loss(logits, labels, pos_weight):
    mask = labels >= 0
    target = labels.clamp(0, 1)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, target,
        pos_weight=pos_weight, reduction='none')
    return loss[mask].mean()


speech_frames = sum(s['labels'].sum() for s in train_sessions)
all_frames = sum(len(s['labels']) for s in train_sessions)
pos_weight = torch.tensor([(all_frames - speech_frames) / max(speech_frames, 1)], device=DEVICE)
model = CNNBiLSTM().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,} | pos_weight: {pos_weight.item():.3f}')

In [ ]:
@torch.no_grad()
def collect_outputs(loader):
    model.eval(); losses, probs, targets = [], [], []
    for x, y, _ in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP): logits = model(x)
        losses.append(masked_loss(logits, y, pos_weight).item())
        mask = y >= 0; probs.append(torch.sigmoid(logits)[mask].float().cpu().numpy()); targets.append(y[mask].cpu().numpy())
    return float(np.mean(losses)), np.concatenate(probs), np.concatenate(targets).astype(int)


def f1_at_threshold(probs, targets, threshold):
    pred = (probs >= threshold).astype(int)
    return precision_recall_fscore_support(targets, pred, average='binary', zero_division=0)[2]


history, best_f1, best_state, wait = [], -1, None, 0
for epoch in range(1, EPOCHS + 1):
    model.train(); train_losses = []
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP): loss = masked_loss(model(x), y, pos_weight)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer); scaler.update(); train_losses.append(loss.item())
    val_loss, val_probs, val_targets = collect_outputs(val_loader)
    val_f1 = f1_at_threshold(val_probs, val_targets, 0.5)
    scheduler.step(val_f1); history.append((epoch, np.mean(train_losses), val_loss, val_f1))
    print(f'Epoch {epoch:02d} | train {np.mean(train_losses):.4f} | val {val_loss:.4f} | val F1 {val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1, wait = val_f1, 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({'model_state': best_state, 'config': {'n_mels': N_MELS, 'sr': SAMPLE_RATE, 'hop': HOP_LENGTH}}, WORK_ROOT / 'cnn_bilstm_vad_best.pt')
    else: wait += 1
    if wait >= PATIENCE: print('Early stopping.'); break
history = pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_loss', 'val_f1'])

In [ ]:
model.load_state_dict(best_state)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.epoch, history.train_loss, label='train'); ax[0].plot(history.epoch, history.val_loss, label='validation'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history.epoch, history.val_f1, label='validation F1'); ax[1].set_title('Validation F1'); ax[1].legend()
plt.tight_layout(); plt.savefig(WORK_ROOT / 'training_curves.png', dpi=150); plt.show()


_, vp, vy = collect_outputs(val_loader)
thresholds = np.linspace(0.1, 0.9, 81)
best_threshold = max(thresholds, key=lambda t: f1_at_threshold(vp, vy, t))
print(f'Validation threshold: {best_threshold:.2f} | F1: {f1_at_threshold(vp, vy, best_threshold):.4f}')

In [ ]:
_, test_probs, test_targets = collect_outputs(test_loader)
test_pred = (test_probs >= best_threshold).astype(int)
p, r, f1, _ = precision_recall_fscore_support(test_targets, test_pred, average='binary', zero_division=0)
metrics = {'threshold': float(best_threshold), 'precision': float(p), 'recall': float(r), 'f1': float(f1),
           'accuracy': float(accuracy_score(test_targets, test_pred)),
           'balanced_accuracy': float(balanced_accuracy_score(test_targets, test_pred))}
print(pd.Series(metrics).to_string())
print('\nClassification report:\n', classification_report(test_targets, test_pred, target_names=['pause', 'speech'], zero_division=0))
plt.figure(figsize=(4, 3)); sns.heatmap(confusion_matrix(test_targets, test_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['pause', 'speech'], yticklabels=['pause', 'speech']); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()
with open(WORK_ROOT / 'test_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)

In [ ]:
# Qualitative timeline for a few held-out sessions.
model.eval()
for s in test_sessions[:3]:
    x = torch.from_numpy(s['mel'][None, None]).float().to(DEVICE)
    with torch.no_grad(): pred = torch.sigmoid(model(x))[0].cpu().numpy() >= best_threshold
    n = min(len(s['labels']), len(pred)); t = np.arange(n) * HOP_LENGTH / SAMPLE_RATE
    plt.figure(figsize=(14, 2.5)); plt.step(t, s['labels'][:n], where='post', label='actual'); plt.step(t, pred[:n], where='post', label='predicted', alpha=.8)
    plt.ylim(-.1, 1.1); plt.yticks([0, 1], ['pause', 'speech']); plt.xlabel('Time (s)'); plt.title(Path(s['path']).name); plt.legend(loc='upper right'); plt.show()


config = {'sample_rate': SAMPLE_RATE, 'n_mels': N_MELS, 'n_fft': N_FFT, 'hop_length': HOP_LENGTH,
          'crop_frames': CROP_FRAMES, 'eval_frames': EVAL_FRAMES, 'threshold': float(best_threshold),
          'architecture': 'CNN frontend + 2-layer bidirectional LSTM + frame classifier'}
with open(WORK_ROOT / 'preprocessing_config.json', 'w') as f: json.dump(config, f, indent=2)
print('Saved artifacts to:', WORK_ROOT)